# 健康監測 RO：功能驗收與固定設定復測
逐格執行，不要 Run All。新硬體 ID 為 ROH2；每批暖機後先檢查 1024 點，再保存 1024 點。沿用 25 MHz、divider 512。先確認你的獨占時段與舊 kernel 已關閉。這是研究用健康監測候選版本，不是已認證 TRNG。

In [ ]:
print("kernel OK", flush=True)
import json, platform, pynq, zipfile
from pathlib import Path
from entropy_health_pynq import EntropySession, HealthFailure, validate_artifacts, save_capture
from entropy_campaign import collect_campaign, export_datasets
print("PYNQ", pynq.__version__, "Python", platform.python_version())

## 1. 離線檢查與操作資料
確認實際板號及操作日期。校驗失敗不能跳過。這格不操作硬體。

In [ ]:
board_id = "21600"
experiment_date = "2026-11-11"
print(validate_artifacts("entropy_health.bit"))
print("Board", board_id, "Date", experiment_date)

## 2. 載入健康監測電路一次
應看到四個檢查點，最後 RO OFF。不要在這一步載入舊 entropy_ro.bit。

In [ ]:
session = EntropySession("entropy_health.bit")

## 3. 第一段：16 批功能驗收
成功時每批 status=194（0xc2）、startup_seen=1024、observed=2048、count=1024，且三個故障遮罩與 RO enable 都為零。警報即停止並保存診斷；不自動清故障重跑。這段沒有在實板強制注入故障，故障分支以本機模擬驗證。

In [ ]:
from datetime import datetime, timezone
import uuid
functional_folder = Path("health_functional") / (experiment_date + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
functional_folder.mkdir(parents=True, exist_ok=False)
functional_ok = False
try:
    for i in range(16):
        capture = session.capture(divider=512)
        save_capture(capture, functional_folder / ("capture_{:03d}".format(i+1)), board_id)
        print("HEALTH {}/16".format(i+1), capture["health"], flush=True)
    functional_ok = True
except HealthFailure as exc:
    (functional_folder / "fault.json").write_text(json.dumps(exc.snapshot, indent=2), encoding="utf-8")
    raise
finally:
    try:
        session.close()
        print("RO OFF confirmed", flush=True)
    except BaseException as exc:
        (functional_folder / "cleanup_error.json").write_text(json.dumps({"error":repr(exc)}), encoding="utf-8")
        raise
assert functional_ok
print("Functional folder:", str(functional_folder))

## 4. 第二段：1000 批固定設定資料復測
只有第一段成功、RO 關閉且仍在你的使用時段，才明確執行本格。此版本增加了啟動健康取樣，也重新實作，不能沿用舊版 6.65 的估計。每批的 1024 個啟動樣本不輸出；原始 payload 全部保存。本格不重新下載電路。

In [ ]:
assert functional_ok, "先完成第一段"
campaign = collect_campaign(session, board_id, experiment_date, output_root="health_assessment", target_batches=1000)
print(campaign)
assert campaign["status"] == "complete" and campaign["batches"] == 1000
assert campaign["ro_off_confirmed"]

## 5. 離線匯出，不操作硬體
將 payload 匯出聯合資料、逐通道資料，以及每次啟動檢查完成後的前 1000 個可用樣本。不是整板上電矩陣。這格不做熵認證。

In [ ]:
dataset = export_datasets(campaign["folder"])
print(json.dumps(dataset, indent=2))
assert dataset["total_samples"] == 1024000

## 6. 打包下載
下載 ZIP 與有執行輸出的本 notebook。只完成第一段也可執行；若第二段中斷未回傳 campaign，請將畫面印出的 Campaign folder 路徑以 Path 加入 folders，或先提供錯誤畫面協助保存。不要重新跑硬體格。

In [ ]:
folders = [functional_folder]
if "campaign" in globals():
    folders.append(Path(campaign["folder"]))
zip_path = Path("entropy_health_" + functional_folder.name + ".zip")
with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                z.write(str(p), str(Path(folder.parent.name) / folder.name / p.relative_to(folder)))
print("Download:", str(zip_path))
print("No more hardware actions needed")